# Phase 5 — Fine-tune PhoBERT cho phân loại cảm xúc review Điện Máy Xanh

**Mô hình hiện đại** bổ sung cho NB/SVM: `vinai/phobert-base-v2`.

- PhoBERT là công cụ NLP tiếng Việt được giới thiệu ở **Ch.1**. Transformer tạo **vector ngữ cảnh** (**Ch.3**, phần Word Embeddings), khắc phục điểm yếu bỏ qua thứ tự từ của BoW/TF-IDF.
- Việc fine-tune (thêm lớp phân loại, huấn luyện lại toàn mạng) là **[Ngoài slide]**: slide chỉ giới thiệu PhoBERT/Transformer, không hướng dẫn fine-tune.
- Đầu vào là `text_seg` (chỉ chuẩn hóa, giữ hoa thường, dấu câu và stopword). Văn bản được **tách từ lại bằng RDRSegmenter (VnCoreNLP)** như PhoBERT khuyến nghị; nếu không khởi động được VnCoreNLP (thiếu Java) thì dùng luôn kết quả tách từ underthesea có sẵn trong `text_seg` và ghi chú lại.
- Cấu hình: `max_len=128`, `batch=16` (thiếu VRAM thì tự giảm xuống 8 và dùng gradient accumulation 2), fp16, `lr=2e-5`, 4 epoch, cross-entropy có **trọng số lớp theo phân bố train**, giữ epoch có **macro-F1 trên val** cao nhất.
- Tập test **không** dùng để chọn epoch. Notebook chỉ lưu dự đoán trên test; chỉ số trên test được tính ở Phase 6 (`src/evaluate.py`).

**Chạy trên máy:** mở notebook từ thư mục `notebooks/` của dự án (cần GPU).
**Chạy trên Colab:** Runtime → GPU, chạy từ trên xuống; notebook tự cài thư viện và yêu cầu upload `train.csv`, `val.csv`, `test.csv` (trong `data/processed/`).

Đầu ra: `results/metrics/phobert.json`, `results/predictions/phobert_test.csv`, `results/predictions/phobert_val.csv`.

**Seed:** mặc định 42 (mô hình dùng cho test). Đặt biến môi trường `PHOBERT_SEED=43`/`44` để chạy lại với seed khác nhằm đo độ dao động; khi đó chỉ ghi `phobert_seed43.json`… và dự đoán val.

## 1. Môi trường

In [1]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "py_vncorenlp"], check=True)
    ROOT = Path("/content")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
SEED = int(os.environ.get("PHOBERT_SEED", 42))  # 42 = model used on test; 43/44 = seed-variance runs
SUFFIX = "" if SEED == 42 else f"_seed{SEED}"
MODEL_DIR = ROOT / "models" / f"phobert_best{SUFFIX}"
for d in (DATA, RESULTS / "metrics", RESULTS / "predictions", MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("IN_COLAB:", IN_COLAB, "| ROOT:", ROOT)

IN_COLAB: False | ROOT: D:\San\ki7\PTKPDLVB\btl1\btl-dmx


In [2]:
if IN_COLAB and not all((DATA / f).exists() for f in ("train.csv", "val.csv", "test.csv")):
    from google.colab import files
    print("Upload train.csv, val.csv, test.csv (từ data/processed/ của dự án)")
    for name, content in files.upload().items():
        (DATA / name).write_bytes(content)
print(sorted(p.name for p in DATA.glob("*.csv")))

['test.csv', 'train.csv', 'val.csv']


In [3]:
import random, time, json, copy
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, DataCollatorWithPadding
from sklearn.metrics import f1_score, accuracy_score

CFG = dict(model="vinai/phobert-base-v2", max_len=128, batch=16, accum=1, lr=2e-5, epochs=4,
           weight_decay=0.01, warmup_ratio=0.1, fp16=True, seed=SEED)
LABELS = ["Negative", "Neutral", "Positive"]
L2I = {l: i for i, l in enumerate(LABELS)}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
print(device, GPU, "| torch", torch.__version__)

C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda NVIDIA GeForce RTX 3060 Laptop GPU | torch 2.7.1+cu118


## 2. Dữ liệu và tách từ (RDRSegmenter)

In [4]:
train = pd.read_csv(DATA / "train.csv"); val = pd.read_csv(DATA / "val.csv"); test = pd.read_csv(DATA / "test.csv")
for d in (train, val, test):
    d["text_seg"] = d["text_seg"].fillna("")
print({k: len(v) for k, v in dict(train=train, val=val, test=test).items()})
print("train:", train["label"].value_counts().to_dict())

{'train': 5134, 'val': 906, 'test': 500}
train: {'Positive': 3080, 'Negative': 1274, 'Neutral': 780}


In [5]:
SEGMENTER = "underthesea (fallback)"
try:
    import py_vncorenlp
    vn_dir = (ROOT / "models" / "vncorenlp").resolve()
    vn_dir.mkdir(parents=True, exist_ok=True)
    if not (vn_dir / "VnCoreNLP-1.2.jar").exists():
        py_vncorenlp.download_model(save_dir=str(vn_dir))  # uses wget (available on Colab)
    cwd = os.getcwd()
    rdr = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(vn_dir))
    os.chdir(cwd)  # VnCoreNLP changes the working directory
    def resegment(t):
        raw = t.replace("_", " ").strip()   # undo underthesea segmentation
        return " ".join(rdr.word_segment(raw)) if raw else ""
    SEGMENTER = "RDRSegmenter (VnCoreNLP)"
except Exception as e:
    print("VnCoreNLP unavailable -> keep underthesea segmentation:", repr(e)[:200])
    resegment = lambda t: t

t0 = time.time()
for d in (train, val, test):
    d["x"] = d["text_seg"].map(resegment)
print(SEGMENTER, f"({time.time() - t0:.0f}s)")
for a, b in zip(train["text_seg"][:3], train["x"][:3]):
    print(" underthesea:", a[:100]); print(" input      :", b[:100])

RDRSegmenter (VnCoreNLP) (2s)
 underthesea: Nấu cơm bình_thường , được
 input      : Nấu cơm bình_thường , được
 underthesea: Máy dùng tốt . Tiếng không bị to quá . Mỗi tội hút tóc kém
 input      : Máy dùng tốt . Tiếng không bị to quá . Mỗi_tội hút tóc kém
 underthesea: Thợ điện_máy xanh nắp cho mình Model Cục nóng khác model dàn lạnh , chán hết chỗ nói . Mua máy model
 input      : Thợ_điện máy xanh nắp cho mình Model_Cục nóng khác model dàn lạnh , chán hết_chỗ_nói . Mua máy model


## 3. Tokenize, trọng số lớp

In [6]:
tok = AutoTokenizer.from_pretrained(CFG["model"])

def encode(df):
    enc = tok(list(df["x"]), truncation=True, max_length=CFG["max_len"])
    return [{"input_ids": i, "attention_mask": m, "labels": L2I[l]}
            for i, m, l in zip(enc["input_ids"], enc["attention_mask"], df["label"])]

ds = {k: encode(v) for k, v in dict(train=train, val=val, test=test).items()}
lens = [len(tok(t)["input_ids"]) for t in train["x"]]
TRUNC = float(np.mean(np.array(lens) > CFG["max_len"]))
print(f"train tokens: median {np.median(lens):.0f}, p95 {np.percentile(lens, 95):.0f}, truncated {TRUNC:.1%}")

counts = train["label"].value_counts().reindex(LABELS).values
class_w = torch.tensor(len(train) / (len(LABELS) * counts), dtype=torch.float)
print("class weights:", dict(zip(LABELS, class_w.round(decimals=3).tolist())))
collate = DataCollatorWithPadding(tok)

C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--vinai--phobert-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train tokens: median 15, p95 74, truncated 1.5%
class weights: {'Negative': 1.343000054359436, 'Neutral': 2.194000005722046, 'Positive': 0.5559999942779541}


## 4. Fine-tune (chọn epoch theo macro-F1 trên val)

In [7]:
@torch.no_grad()
def predict(model, data, bs=64):
    model.eval(); probs = []
    for batch in DataLoader(data, batch_size=bs, collate_fn=collate):
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        with torch.autocast(device.type, dtype=torch.float16, enabled=CFG["fp16"] and device.type == "cuda"):
            logits = model(**batch).logits
        probs.append(torch.softmax(logits.float(), -1).cpu())
    return torch.cat(probs).numpy()

def val_scores(y, pred):
    f1 = f1_score(y, pred, labels=range(3), average=None, zero_division=0)
    return {"macro_f1": round(float(f1.mean()), 4), "accuracy": round(float(accuracy_score(y, pred)), 4),
            **{f"f1_{l}": round(float(v), 4) for l, v in zip(LABELS, f1)}}

def train_run(batch, accum):
    set_seed(CFG["seed"])
    model = AutoModelForSequenceClassification.from_pretrained(
        CFG["model"], num_labels=3, id2label=dict(enumerate(LABELS)), label2id=L2I).to(device)
    g = torch.Generator().manual_seed(CFG["seed"])
    loader = DataLoader(ds["train"], batch_size=batch, shuffle=True, collate_fn=collate, generator=g)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    steps = (len(loader) // accum) * CFG["epochs"]
    sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup_ratio"] * steps), steps)
    scaler = torch.amp.GradScaler(enabled=CFG["fp16"] and device.type == "cuda")
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_w.to(device))
    y_val = np.array([d["labels"] for d in ds["val"]])
    history, best = [], {"macro_f1": -1}
    for ep in range(1, CFG["epochs"] + 1):
        model.train(); t0 = time.time(); tot = 0.0
        for i, b in enumerate(loader):
            b = {k: v.to(device) for k, v in b.items()}
            labels = b.pop("labels")
            with torch.autocast(device.type, dtype=torch.float16, enabled=CFG["fp16"] and device.type == "cuda"):
                loss = loss_fn(model(**b).logits.float(), labels) / accum
            scaler.scale(loss).backward()
            if (i + 1) % accum == 0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
            tot += loss.item() * accum
        s = val_scores(y_val, predict(model, ds["val"]).argmax(1))
        rec = {"epoch": ep, "train_loss": round(tot / len(loader), 4), "sec": round(time.time() - t0), **{f"val_{k}": v for k, v in s.items()}}
        history.append(rec); print(rec)
        if s["macro_f1"] > best["macro_f1"]:
            best = {**s, "epoch": ep}
            model.save_pretrained(MODEL_DIR); tok.save_pretrained(MODEL_DIR)
    return history, best

try:
    BATCH, ACCUM = CFG["batch"], CFG["accum"]
    history, best = train_run(BATCH, ACCUM)
except torch.cuda.OutOfMemoryError:
    print("CUDA OOM with batch 16 -> batch 8 + gradient accumulation 2")
    torch.cuda.empty_cache()
    BATCH, ACCUM = 8, 2
    history, best = train_run(BATCH, ACCUM)
print("best epoch:", best)

C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 27912.91it/s]


[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'epoch': 1, 'train_loss': 0.7613, 'sec': 46, 'val_macro_f1': 0.7356, 'val_accuracy': 0.8179, 'val_f1_Negative': 0.815, 'val_f1_Neutral': 0.4792, 'val_f1_Positive': 0.9128}


...

{'epoch': 3, 'train_loss': 0.41, 'sec': 45, 'val_macro_f1': 0.7813, 'val_accuracy': 0.8532, 'val_f1_Negative': 0.8254, 'val_f1_Neutral': 0.5813, 'val_f1_Positive': 0.9372}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.28it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.27it/s]

{'epoch': 4, 'train_loss': 0.3247, 'sec': 47, 'val_macro_f1': 0.7716, 'val_accuracy': 0.8455, 'val_f1_Negative': 0.8055, 'val_f1_Neutral': 0.5724, 'val_f1_Positive': 0.9369}
best epoch: {'macro_f1': 0.7813, 'accuracy': 0.8532, 'f1_Negative': 0.8254, 'f1_Neutral': 0.5813, 'f1_Positive': 0.9372, 'epoch': 3}


## 5. Dự đoán val/test với checkpoint tốt nhất và lưu kết quả

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
out = {}
for name, df in (("val", val), ("test", test)) if SEED == 42 else (("val", val),):
    p = predict(model, ds[name])
    pred = pd.DataFrame({"review_id": df["review_id"], "label": df["label"],
                         "pred": [LABELS[i] for i in p.argmax(1)],
                         **{f"prob_{l}": p[:, i].round(4) for i, l in enumerate(LABELS)}})
    pred.to_csv(RESULTS / "predictions" / f"phobert{SUFFIX}_{name}.csv", index=False, encoding="utf-8")
    out[name] = pred
check = val_scores(out["val"]["label"].map(L2I), out["val"]["pred"].map(L2I))
assert abs(check["macro_f1"] - best["macro_f1"]) < 1e-3, check

meta = {"model": CFG["model"], "config": {**CFG, "batch_used": BATCH, "grad_accum_used": ACCUM},
        "segmenter": SEGMENTER, "input": "text_seg (re-segmented)", "truncated_train_pct": round(TRUNC, 4),
        "class_weights": dict(zip(LABELS, [round(float(w), 4) for w in class_w])),
        "gpu": GPU, "in_colab": IN_COLAB, "history": history, "best_epoch": best["epoch"],
        "val_best": {k: v for k, v in best.items() if k != "epoch"},
        "note": "epoch chosen on val macro-F1 (val label = star label); test predictions only, metrics in Phase 6"}
(RESULTS / "metrics" / f"phobert{SUFFIX}.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: meta[k] for k in ("segmenter", "best_epoch", "val_best")}, ensure_ascii=False, indent=1))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   4%|▍         | 9/201 [00:00<00:02, 81.99it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1625.80it/s]

{
 "segmenter": "RDRSegmenter (VnCoreNLP)",
 "best_epoch": 3,
 "val_best": {
  "macro_f1": 0.7813,
  "accuracy": 0.8532,
  "f1_Negative": 0.8254,
  "f1_Neutral": 0.5813,
  "f1_Positive": 0.9372
 }
}


In [9]:
if IN_COLAB:  # download results to put back into the project
    import shutil
    from google.colab import files
    shutil.make_archive("/content/phobert_results", "zip", RESULTS)
    files.download("/content/phobert_results.zip")